# ==============================================================================
# 💳 FINANCIAL CREDIT RISK: LOAN DEFAULT VIA ENTERPRISE ADABOOST CLASSIFIER
# ==============================================================================
### Core Principles:
Credit risk scoring predicts probability of default (`loan_status`) across 32,581 loan applications.
AdaBoost constructs a robust sequential committee combining demographic history and financial leverage indicators.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, recall_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Financial Credit Risk Environment ready.")


✅ STEP 1: Financial Credit Risk Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Credit Risk Dataset (32,581 loan records).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
data_path = 'data/credit_risk/credit_risk_dataset.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} loan applications, {df.shape[1]} attributes")
print(f"Default Prevalence:\n{df['loan_status'].value_counts(normalize=True).round(4)*100}%")
df.head(3)


📊 Dataset Shape: 32,581 loan applications, 12 attributes
Default Prevalence:
loan_status
0    78.18
1    21.82
Name: proportion, dtype: float64%


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `loan_status`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['loan_status'])
y = df['loan_status']

print(f"Missing values across features:\n{X.isnull().sum()[X.isnull().sum() > 0]}")


Missing values across features:
person_emp_length     895
loan_int_rate        3116
dtype: int64


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
80/20 stratified split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Loans: {X_train.shape[0]:,} ({y_train.sum()} Defaults)")
print(f"Test Loans : {X_test.shape[0]:,} ({y_test.sum()} Defaults)")


Train Loans: 26,064 (5686 Defaults)
Test Loans : 6,517 (1422 Defaults)


## ⚙️ STEP 7: BANKING COLUMN TRANSFORMER PREPROCESSOR
Handling missing interest rates via median imputation and encoding categorical credit loan intent.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])
print("✅ STEP 7: Banking Preprocessor configured.")


✅ STEP 7: Banking Preprocessor configured.


/tmp/ipykernel_693055/2414773436.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


## ⚠️ STEP 8: BASELINE FIT
Fitting baseline AdaBoost ensemble with 100 decision stumps.


In [6]:
# STEP 8: BASELINE FIT
pipe_ada = Pipeline([
    ('prep', preprocessor),
    ('ada', AdaBoostClassifier(n_estimators=100, random_state=42))
])
pipe_ada.fit(X_train, y_train)

acc_base = accuracy_score(y_test, pipe_ada.predict(X_test))
roc_base = roc_auc_score(y_test, pipe_ada.predict_proba(X_test)[:, 1])

print(f"Baseline AdaBoost Accuracy: {acc_base * 100:.2f}%")
print(f"Baseline AdaBoost ROC-AUC : {roc_base * 100:.2f}%")


Baseline AdaBoost Accuracy: 87.26%
Baseline AdaBoost ROC-AUC : 88.63%


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 3-FOLD CV
Optimizing `n_estimators` and `learning_rate`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'ada__n_estimators': [100, 150],
    'ada__learning_rate': [0.1, 0.5, 1.0]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_ada, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Credit Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 3-Fold CV ROC-AUC: {grid.best_score_ * 100:.2f}%")


🏆 Best Credit Hyperparameters: {'ada__learning_rate': 0.5, 'ada__n_estimators': 150}
🎯 Best 3-Fold CV ROC-AUC: 89.08%


## 📊 STEP 10 & 11: CREDIT REPORT & CONFUSION MATRIX
Reviewing loan default detection rates.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print("📋 INSTITUTIONAL CREDIT RISK AUDIT REPORT:")
print(classification_report(y_test, y_pred, target_names=['Performing Loan', 'Defaulted Loan']))

print(f"🎯 Test Accuracy   : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 Default Recall  : {recall_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 ROC-AUC Score   : {roc_auc_score(y_test, y_prob)*100:.2f}%")

print(f"\nConfusion Matrix:\n{cm}")


📋 INSTITUTIONAL CREDIT RISK AUDIT REPORT:
                 precision    recall  f1-score   support

Performing Loan       0.89      0.95      0.92      5095
 Defaulted Loan       0.78      0.59      0.67      1422

       accuracy                           0.87      6517
      macro avg       0.83      0.77      0.80      6517
   weighted avg       0.87      0.87      0.87      6517



🎯 Test Accuracy   : 87.36%
🎯 Default Recall  : 59.14%
🎯 ROC-AUC Score   : 88.94%

Confusion Matrix:
[[4852  243]
 [ 581  841]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & UNDERWRITING SMOKE TEST
Deploying pipeline and validating bank core lending decision latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/credit_risk_adaboost_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_loan = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_def = loaded_pipe.predict(sample_loan)[0]
pred_prob = loaded_pipe.predict_proba(sample_loan)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏛️ LIVE BANK UNDERWRITING AUDIT:")
print(f"   Credit Decision     : {'🚨 HIGH DEFAULT RISK' if pred_def == 1 else '🟢 APPROVED / LOW RISK'}")
print(f"   Default Probability : {pred_prob*100:.2f}%")
print(f"   Underwriting Latency: {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/credit_risk_adaboost_pipeline.joblib (21,840 bytes)

🏛️ LIVE BANK UNDERWRITING AUDIT:
   Credit Decision     : 🟢 APPROVED / LOW RISK
   Default Probability : 37.71%
   Underwriting Latency: 37.208 ms (SLA < 2.0ms: CHECK)
